# IMU measurements

An IMU usually provides two types of measurements.

Gyroscope:
$$
\boldsymbol{\omega}_m=\boldsymbol{\omega}+\mathbf b_g+\mathbf n_g
$$

Accelerometer:
$$
\mathbf a_m=R_{WB}^{T}(\mathbf a_W-\mathbf g_W)+\mathbf b_a+\mathbf n_a
$$

Where:
- $\boldsymbol{\omega}_m$: measured angular velocity
- $\mathbf a_m$: measured specific force (acceleration excluding gravity)
- $\mathbf b_g,\mathbf b_a$: gyroscope and accelerometer biases
- $\mathbf n_g,\mathbf n_a$: measurement noise
- $R_{WB}$: rotation from IMU/body coordinates to world coordinates
- $\mathbf g_W$: gravity vector
We use these measurements to propagate the system state forward.

Visual-inertial SLAM maintains the following state:
$$
\boxed{\mathbf x_k=(R_k,\mathbf p_k,\mathbf v_k,\mathbf b_{g,k},\mathbf b_{a,k})}
$$

It contains rotation, position, velocity, and both sensor biases.

# IMU camera motion prediction

Suppose we have an estimated state at time $t_k$, and new IMU measurements arrive.

For simplicity, consider a small time interval $\Delta t$.

Rotation update:
$$
R_{k+1}=R_k\operatorname{Exp}((\boldsymbol{\omega}_m-\mathbf b_g)\Delta t)
$$

Acceleration in world coordinates:
$$
\mathbf a_W=R_k(\mathbf a_m-\mathbf b_a)+\mathbf g_W
$$

Velocity update:
$$
\mathbf v_{k+1}=\mathbf v_k+\mathbf a_W\Delta t
$$

Position update:
$$
\mathbf p_{k+1}=\mathbf p_k+\mathbf v_k\Delta t+\frac12\mathbf a_W\Delta t^2
$$

Note: These are simplified discrete integration equations. Practical implementations use more careful integration and uncertainty propagation.

Note: the IMU does not directly measure position or velocity. We must integrate acceleration to obtain velocity and integrate again to obtain position.

This causes drift, particularly because accelerometer biases accumulate into substantial position errors.

# IMU pre-integration

Between two consecutive camera frames, we receive approximately 6–7 IMU measurements. Between two keyframes, we may receive hundreds.

Preintegration summarizes all IMU measurements between two keyframes into:
$$
\boxed{\Delta R_{ij},\quad\Delta\mathbf v_{ij},\quad\Delta\mathbf p_{ij}}
$$

These represent relative rotation, velocity change, and position change in a gravity-compensated formulation.

Preintegration also propagates covariance and supports corrections when bias estimates change.

# Combine Visual & IMU information

In regular SLAM Bundle Adjustment minimizes visual reprojection errors:

$$
E_{\text{visual}}=\sum_{i,j}\|\mathbf u_{ij}-\pi(T_i,\mathbf P_j)\|^2
$$

In Visual-Inertial SLAM, we additionally introduce IMU constraints:
$$
\boxed{
E_{\text{total}}=
E_{\text{visual}}+
E_{\text{IMU}}+
E_{\text{bias}}+
E_{\text{prior}}
}
$$

Here:
- $E_{\text{visual}}$ penalizes disagreement between observed and predicted feature positions.
- $E_{\text{IMU}}$ penalizes disagreement between predicted motion and preintegrated IMU measurements.
- $E_{\text{bias}}$ models how accelerometer and gyroscope biases evolve.
- $E_{\text{prior}}$ incorporates prior information and constraints from marginalized states.

In practice, residuals are weighted by their uncertainty, rather than simply added with equal importance.

# Initialization

Before reliable visual-inertial tracking, we need to establish:
1. Camera-to-IMU extrinsic calibration (relative rotation and translation).
2. Camera–IMU timestamp synchronization.
3. Initial gyroscope and accelerometer biases.
4. Gravity direction.
5. Initial velocities.
6. Metric scale, particularly for monocular cameras.

Unlike monocular regular SLAM, monocular-inertial SLAM can provide absolute metric scale.

# Integration approaches

There are two common architectural choices:

1. Filter-based integration (EKF)

    IMU predicts the state at high frequency; visual observations periodically correct it.

    This follows the prediction–correction model we previously discussed for VIO.

2. Optimization-based integration

    IMU measurements become constraints between states in a factor graph, alongside visual observations.

    This is the approach used by ORB-SLAM3.